# 02 — Inventory & Calculation

**Rules:** PEP ecopassport PCR ed4 (S08) and PSR-0018 ed1.3 (S01), as set in `01_scope_boundary`.

This notebook turns the decisions in notebook 01 into numbers: the life cycle emissions of one wallbox (declared unit, for CDR sizing), the emissions per kWh supplied (functional unit, for PEP benchmarking), and the share of each stage (hotspots).

**How to use:** Fill in every cell marked `TODO`, in order. Every number must have a source ID (`data/raw/SOURCES.md`) or an assumption ID (`docs/assumptions_log.md`). Calculation cells produce results only after the inputs above them are filled in.

## 0. Parameters from notebook 01

These values were decided in notebook 01. Do not change them here; change notebook 01 first if a decision changes.

In [ ]:
SERVICE_LIFE_YEARS = 10                 # S01, p. 20
ENERGY_PER_CHARGING_POINT_KWH = 28_251  # S01, p. 20 (derivation in notebook 01, Section 2)
CHARGING_POINTS = 1                     # Notebook 01, reference flow
CABLE_LENGTH_FT = 24                    # A02 (sensitivity range 13.1-25 ft)
FT_TO_M = 0.3048                        # Exact unit conversion
CUTOFF_THRESHOLD = 0.05                 # S08, §2.3 (aggregate mass, energy, impact)
DQR_RELEVANCE_SHARE = 0.80              # S08, §2.9.3 (score datasets covering >= 80% of impact)
DEFAULT_DQR = 5                         # S08, §2.9.3 (when quality cannot be assessed)

ENERGY_SUPPLIED_KWH = ENERGY_PER_CHARGING_POINT_KWH * CHARGING_POINTS
REFERENCE_FLOW = 1 / ENERGY_SUPPLIED_KWH  # wallboxes per kWh (functional unit)
CABLE_LENGTH_M = CABLE_LENGTH_FT * FT_TO_M

print(f"Energy supplied over service life: {ENERGY_SUPPLIED_KWH:,} kWh")
print(f"Reference flow: {REFERENCE_FLOW:.4e} wallboxes per kWh")
print(f"Cable length: {CABLE_LENGTH_M:.2f} m")

## 1. Data Sources

**Guiding questions**
- Which public databases or datasets can provide emission factors for the materials in a wallbox? (ecoinvent requires a paid license, so it cannot be used.)
- Where does each candidate rank in the PCR ed4 source priority order (S08, §2.9)?
- Which candidate offers global average (GLO) datasets, as required by A03?

In [1]:
import pandas as pd

data_sources = pd.DataFrame(
    [
        {
            "source_id": "S09",
            "name": "Copper and Copper Alloy Semi-Fabricated Products LCA",
            "publisher": "International Copper Association (ICA)",
            "public_access": True,
            "pcr_priority_level": 4,
            "glo_datasets": "Global (excludes China)",
            "notes": "Copper wire rod; ends before wire drawing, which must be added separately",
        },
    ]
)
data_sources

,source_id,name,publisher,public_access,pcr_priority_level,glo_datasets,notes
0,S09,Copper and Copper Alloy Semi-Fabricated Produc...,International Copper Association (ICA),True,4,Global (excludes China),"Copper wire rod; ends before wire drawing, whi..."


## 2. Reference Product and Bill of Materials (BOM)

With public data only, the BOM is a **proxy BOM** built with a data hierarchy:

| Tier | Data | Use in this project |
|---|---|---|
| 1 | Manufacturer's actual BOM | Not available (public data only) |
| 2 | Material composition from published PEPs or EPDs of similar wallboxes | Main source of material shares |
| 3 | Engineering estimate | Cable copper from length × gauge |
| 4 | Published LCA studies of EV chargers | Fill remaining gaps |
| 5 | Spend-based estimate | Screening only, last resort |

**Guiding questions**
- Which reference product has a public total mass (S04–S06)? Does that mass include the cable and the mounting bracket?
- Which published PEP or EPD of a wallbox can provide material shares?
- Which PCR ed4 Table 2 items (copper, gold, silver, microprocessors) appear, and where?
- Does the sum of all components equal the total product mass (mass balance)?

In [ ]:
# TODO: one row per component. data_tier follows the table above; table2_item marks PCR ed4 Table 2 materials.
bom = pd.DataFrame(
    columns=["component", "material", "mass_kg", "source_id", "data_tier", "table2_item"]
)

# TODO: total mass of the reference product, with its source ID
REFERENCE_PRODUCT_MASS_KG = None

if REFERENCE_PRODUCT_MASS_KG is not None and not bom.empty:
    bom_total = bom["mass_kg"].sum()
    gap = REFERENCE_PRODUCT_MASS_KG - bom_total
    print(f"BOM total: {bom_total:.3f} kg | Product mass: {REFERENCE_PRODUCT_MASS_KG:.3f} kg | Gap: {gap:.3f} kg ({gap / REFERENCE_PRODUCT_MASS_KG:.1%})")
bom

## 3. Emission Factors and Data Quality (DQR)

**Guiding questions**
- For each material in the BOM, which dataset gives its emission factor (kg CO₂e per kg)?
- How well does each dataset match in technology (TeR), geography (GR), and time (TiR), on a scale from 1 (best) to 5 (worst)?
- Which datasets together cover at least 80% of the climate change impact, so they must be scored?

In [ ]:
# TODO: one row per material. Scores follow PCR ed4 Annex E (S08); use DEFAULT_DQR when quality cannot be assessed.
emission_factors = pd.DataFrame(
    columns=["material", "ef_kgco2e_per_kg", "dataset", "source_id", "TeR", "GR", "TiR"]
)

if not emission_factors.empty:
    emission_factors["DQR"] = emission_factors[["TeR", "GR", "TiR"]].mean(axis=1)
emission_factors

## 4. Manufacturing Stage (A1–A3)

Emissions = mass of each material × its emission factor.

In [ ]:
manufacturing = pd.DataFrame()
if not bom.empty and not emission_factors.empty:
    manufacturing = bom.merge(emission_factors[["material", "ef_kgco2e_per_kg"]], on="material", how="left")
    missing = manufacturing[manufacturing["ef_kgco2e_per_kg"].isna()]["material"].unique()
    if len(missing):
        print("Materials without an emission factor:", list(missing))
    manufacturing["kgco2e"] = manufacturing["mass_kg"] * manufacturing["ef_kgco2e_per_kg"]
    print(f"Manufacturing total: {manufacturing['kgco2e'].sum():.2f} kg CO2e per wallbox")
manufacturing

## 5. Distribution (A4) and Installation (A5)

**Guiding questions**
- What default transport scenario does PCR ed4 set for distribution (S08)?
- PSR-0018 sets packaging waste treatment rates and a 1,000 km transport distance for installation (S01, pp. 17–18). How do they apply to the packaging in your BOM?

In [ ]:
# TODO: calculate from the PCR / PSR default scenarios and record each input's source ID
DISTRIBUTION_KGCO2E = None
INSTALLATION_KGCO2E = None

## 6. Use Stage (B1–B7)

PSR-0018 (S01, §3.2.4) counts only the charger's own consumption; electricity delivered to the vehicle is excluded.

- Conduction losses = R × I² × T × X / 1000 (kWh)
- Intrinsic consumption = (P_active × %T_active + P_standby_active × %T_standby_active + P_standby_passive × %T_standby_passive + P_off × %T_off) × service life

**Guiding questions**
- Which inputs come from the PSR scenario (S01, p. 20; S02), and which must come from the reference product's datasheet?
- Which EPA eGRID emission factors (kg CO₂e per kWh) represent California and Texas, and which eGRID year?

In [ ]:
# TODO: fill in each input with its source ID
R_OHM = None            # resistance of the conductive path
I_AMP = None            # charging current
T_HOURS = None          # charge time per session
X_CHARGES = None        # number of charges over the service life
INTRINSIC_KWH = None    # intrinsic consumption over the service life

GRID_EF_KGCO2E_PER_KWH = {"California": None, "Texas": None}  # EPA eGRID

use_stage = {}
if None not in (R_OHM, I_AMP, T_HOURS, X_CHARGES, INTRINSIC_KWH):
    losses_kwh = R_OHM * I_AMP**2 * T_HOURS * X_CHARGES / 1000
    use_kwh = losses_kwh + INTRINSIC_KWH
    print(f"Use-stage electricity: {use_kwh:.1f} kWh over {SERVICE_LIFE_YEARS} years")
    for state, ef in GRID_EF_KGCO2E_PER_KWH.items():
        if ef is not None:
            use_stage[state] = use_kwh * ef
use_stage

## 7. End of Life (C1–C4)

**Guiding questions**
- What default end-of-life scenario does PCR ed4 set for electrical and electronic equipment (S08)?
- Which materials in the BOM are recycled, incinerated, or landfilled under that scenario?

In [ ]:
# TODO: calculate from the PCR default end-of-life scenario and record each input's source ID
END_OF_LIFE_KGCO2E = None

## 8. Results

- **Per wallbox (declared unit):** sizes the CDR plan in notebook 04
- **Per kWh (functional unit):** benchmarks against published PEPs in notebook 03
- **Share by stage:** identifies the hotspots

In [ ]:
stage_values = {
    "Manufacturing (A1-A3)": manufacturing["kgco2e"].sum() if "kgco2e" in manufacturing else None,
    "Distribution (A4)": DISTRIBUTION_KGCO2E,
    "Installation (A5)": INSTALLATION_KGCO2E,
    "End of life (C1-C4)": END_OF_LIFE_KGCO2E,
}

results = {}
if None not in stage_values.values():
    for state, use_kgco2e in use_stage.items():
        stages = {**stage_values, "Use (B1-B7)": use_kgco2e}
        total = sum(stages.values())
        results[state] = pd.DataFrame(
            {
                "kgco2e_per_wallbox": stages,
                "kgco2e_per_kwh": {k: v * REFERENCE_FLOW for k, v in stages.items()},
                "share": {k: v / total for k, v in stages.items()},
            }
        )
        print(f"{state}: {total:.1f} kg CO2e per wallbox | {total * REFERENCE_FLOW:.2e} kg CO2e per kWh")
results

## 9. Cut-off Compliance

Notebook 01, Section 4: excluded flows must stay at or below 5% of mass, primary energy, and environmental impact in aggregate (S08, §2.3). PCR ed4 Table 2 items cannot be excluded.

**Guiding questions**
- Which items were left out of the BOM or the stage calculations, and why?
- How was each excluded item's mass, energy, and impact estimated?

In [ ]:
# TODO: one row per excluded item
exclusions = pd.DataFrame(
    columns=["item", "stage", "mass_kg", "energy_mj", "kgco2e", "reason", "source_id"]
)

if not exclusions.empty and REFERENCE_PRODUCT_MASS_KG:
    mass_share = exclusions["mass_kg"].sum() / REFERENCE_PRODUCT_MASS_KG
    print(f"Excluded mass share: {mass_share:.1%} (threshold {CUTOFF_THRESHOLD:.0%})")
exclusions

## 10. Consistency Check (before moving to notebook 03)

Review this notebook as if you were the assurance reviewer.

- [ ] The parameters in Section 0 match notebook 01.
- [ ] Every BOM row has a source ID and a data tier, and the mass balance gap is explained.
- [ ] All PCR ed4 Table 2 items in the product are included.
- [ ] Every emission factor has a source ID and a DQR score.
- [ ] Electricity delivered to the vehicle is excluded from the use stage.
- [ ] Excluded items stay within the 5% cut-off thresholds.
- [ ] Every number is either sourced in `data/raw/SOURCES.md` or logged in `docs/assumptions_log.md`.
- [ ] The README status has been updated.

## References

Full details (publisher, URL, version, access date) are in [`data/raw/SOURCES.md`](../data/raw/SOURCES.md). Assumptions are in [`docs/assumptions_log.md`](../docs/assumptions_log.md).

| ID | Source | Used for |
|---|---|---|
| S01 | P.E.P. Association, PSR-0018 Electric Vehicle Charging Infrastructures, ed1.3 (2025-04-08) | Service life, energy supplied, use-stage formulas, installation defaults |
| S08 | P.E.P. Association, PEP-PCR ed4 (2021-09-06) | Cut-off threshold, DQR method |

<!-- TODO: add each source as it is used in this notebook -->